# Day 4: FastAPI + Pydantic (Building Strongly-Typed REST APIs)

Welcome to Day 4 of **Learn Python in 5 Days**.

## What You Will Learn Today
- Understanding Pydantic v2 **`BaseModel`**, field validation, and data serialization
- Adding constraints and documentation with **`Field(...)`**
- The FastAPI application object, decorators (`@app.get`, `@app.post`), and auto-generated Swagger UI (`/docs`)
- Routing with **Path Parameters** and **Query Parameters**
- Request and Response schema modeling with **`response_model`**
- Granular error translation using **`HTTPException`**
- Testing API endpoints interactively with **`fastapi.testclient.TestClient`**
- Structuring a clean, 3-layer modular API project

---

## 1. Pydantic v2: Runtime Data Validation
In TypeScript, interfaces only exist during compilation. In Python, **Pydantic** validates, parses, and coerces data at **runtime**.

### Core Methods:
- `.model_dump()`: Converts model instance into a Python dictionary
- `.model_dump_json()`: Serializes model directly to a JSON string
- `.model_validate(raw_dict)`: Validates and parses raw dictionary into a model instance

In [ ]:
from pydantic import BaseModel, Field, ValidationError

# 1. Define a Pydantic Schema
class TaskItem(BaseModel):
    id: int
    title: str = Field(min_length=3, max_length=50)
    priority: str = Field(default="Normal", pattern="^(Low|Normal|High|Urgent)$")
    completed: bool = False
    tags: list[str] = []

# 2. Valid Instantiation with Type Coercion (e.g. string "101" -> int 101)
valid_task = TaskItem(
    id="101",  # Automatically coerced to int!
    title="Configure Redis Cache",
    priority="High",
    tags=["backend", "perf"]
)

print("Validated Task:", valid_task)
print("Dumped Dict:", valid_task.model_dump())
print("Dumped JSON:", valid_task.model_dump_json())

### Automatic Validation Rejection
If a client passes an invalid priority string or a title that is too short, Pydantic immediately catches the error and provides structured validation details.

In [ ]:
try:
    invalid_task = TaskItem(
        id=102,
        title="ab",  # Too short! min_length=3
        priority="SuperUrgent"  # Invalid pattern!
    )
except ValidationError as e:
    print("Validation Errors Caught:")
    for err in e.errors():
        print(f" - Field: {err['loc']} | Error: {err['msg']} (Type: {err['type']})")

## 2. Interactive FastAPI Testing with `TestClient`
FastAPI provides `TestClient` (backed by HTTPX). This lets us send real HTTP requests to our API endpoints synchronously inside a notebook without needing to launch an external server!

In [ ]:
from fastapi import FastAPI, HTTPException, status, Query, Path
from fastapi.testclient import TestClient

# 1. Initialize FastAPI app in the notebook
demo_app = FastAPI(title="Notebook Interactive Demo API")

# In-memory dataset
db = [
    {"id": 1, "title": "Setup uv environment", "priority": "High", "completed": True},
    {"id": 2, "title": "Build Pydantic models", "priority": "High", "completed": False},
    {"id": 3, "title": "Refactor CSS styling", "priority": "Low", "completed": False}
]

# 2. Declare Routes
@demo_app.get("/tasks", status_code=status.HTTP_200_OK)
def list_all_tasks(priority: str | None = None, limit: int = 10):
    results = db
    if priority:
        results = [t for t in results if t["priority"].lower() == priority.lower()]
    return results[:limit]

@demo_app.get("/tasks/{task_id}")
def get_single_task(task_id: int = Path(..., ge=1)):
    for t in db:
        if t["id"] == task_id:
            return t
    raise HTTPException(status_code=404, detail=f"Task #{task_id} not found")

# 3. Create TestClient
client = TestClient(demo_app)

# Test GET /tasks
res = client.get("/tasks?priority=High")
print("GET /tasks?priority=High Status:", res.status_code)
print("Response JSON:", res.json())

## 3. Testing POST Requests & 422 Validation Responses
Let's add a `POST /tasks` endpoint that accepts our `TaskItem` schema.

In [ ]:
@demo_app.post("/tasks", status_code=status.HTTP_201_CREATED)
def create_task_endpoint(task_in: TaskItem):
    new_record = task_in.model_dump()
    db.append(new_record)
    return new_record

# 1. Test valid POST request
valid_payload = {"id": 4, "title": "Write automated tests", "priority": "Normal"}
post_res = client.post("/tasks", json=valid_payload)
print("Valid POST Status:", post_res.status_code)
print("Created Record:", post_res.json())

# 2. Test INVALID POST request (Triggers automatic 422 Unprocessable Entity!)
invalid_payload = {"id": "not-a-number", "title": "x", "priority": "Unknown"}
error_res = client.post("/tasks", json=invalid_payload)
print("\nInvalid POST Status:", error_res.status_code)  # 422
print("Automatic Error Detail:", error_res.json()["detail"])

## 4. Testing 404 Error Translation
When an ID does not exist in the database, `HTTPException(status_code=404)` cleanly returns a standard JSON error response.

In [ ]:
res_404 = client.get("/tasks/999")
print("Status Code:", res_404.status_code)
print("Response Body:", res_404.json())

---
## 5. Practice Challenge: Task Statistics & Batch Status Update
### Objective:
Extend the API with two new endpoints:
1. **`GET /tasks/stats`**: Returns summary metrics: `total_tasks: int`, `completed_count: int`, `pending_count: int`, and `high_priority_count: int`.
2. **`PATCH /tasks/{task_id}/complete`**: Marks a task's `completed` field as `True` and returns the updated task. Raises HTTP 404 if the task ID is missing.
3. Test both endpoints using `client.get()` and `client.patch()`.

In [ ]:
# TODO: Write your challenge endpoints here...


---
## 6. Challenge Solution

In [ ]:
# Reference Solution

@demo_app.get("/tasks/stats", tags=["Analytics"])
def get_task_statistics():
    total = len(db)
    completed = sum(1 for t in db if t.get("completed"))
    pending = total - completed
    high_prio = sum(1 for t in db if t.get("priority") == "High")
    return {
        "total_tasks": total,
        "completed_count": completed,
        "pending_count": pending,
        "high_priority_count": high_prio
    }

@demo_app.patch("/tasks/{task_id}/complete", status_code=status.HTTP_200_OK)
def complete_task(task_id: int = Path(..., ge=1)):
    for t in db:
        if t["id"] == task_id:
            t["completed"] = True
            return t
    raise HTTPException(status_code=404, detail=f"Task #{task_id} not found")

# Verify with TestClient
stats_res = client.get("/tasks/stats")
print("Stats Response:", stats_res.json())

patch_res = client.patch("/tasks/2/complete")
print("Patched Task #2 Status:", patch_res.status_code, "| Completed:", patch_res.json()["completed"])

---
## 7. Day 4 Recap & Bridge to Day 5

### What We Accomplished Today:
1. **Pydantic Schemas:** Created strongly typed models with `Field(...)` constraints and auto-generated validation errors.
2. **FastAPI Application:** Built clean REST endpoints with path/query parameters, status codes, and `response_model` filtering.
3. **Decoupled Architecture:** Created standalone application modules in `app/models.py`, `app/services.py`, and `app/main.py`.
4. **Interactive Testing:** Explored `/docs` Swagger UI and tested endpoints programmatically with `TestClient`.

### Looking Ahead to Day 5: SQLite & Putting It Together
Tomorrow is the final capstone day:
- Relational database fundamentals & **SQLite** in Python
- Replacing in-memory state with **parameterized SQL CRUD operations**
- Wiring the entire stack: Client -> FastAPI -> Pydantic -> Service -> SQLite
- Introduction to automated testing with **`pytest`**